# Phase 8: Production ML Pipeline

In this phase, we transition from scattered notebook code to a professional, reusable Scikit-Learn Pipeline. This ensures consistency between training and deployment.

In [ ]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Step 98: Load Dataset
df = pd.read_csv("../datasets/predictive_maintenance.csv")

# Step 99: Encode Type Column (Done outside the pipeline for simplicity here)
encoder = LabelEncoder()
df["Type"] = encoder.fit_transform(df["Type"])

# Step 100: Feature/Target Split
X = df.drop(["Machine failure", "UID", "Product ID"], axis=1)
y = df["Machine failure"]

# Step 101: Numerical Columns
numeric_features = X.columns.tolist()
print(f"Numerical Features: {numeric_features}")

### Step 102, 103 & 104: Building the Pipeline

In [ ]:
# Step 102: Numeric Transformer
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Step 103: Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features)
])

# Step 104: Full ML Pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

print("Production ML Pipeline built successfully.")

### Step 105, 106 & 108: Train & Evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 106: Train Pipeline (Handles preprocessing AND training automatically)
model_pipeline.fit(X_train, y_train)

# Step 108: Evaluate
predictions = model_pipeline.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, predictions))

### Step 109: Save FULL Pipeline

In [ ]:
joblib.dump(model_pipeline, "../models/full_pipeline.pkl")
print("Full production pipeline saved to ../models/full_pipeline.pkl")

### Step 110-113: Inference Simulation & Prediction Function

In [ ]:
# Step 110: Load Pipeline
loaded_pipeline = joblib.load("../models/full_pipeline.pkl")

# Step 112: Create Prediction Function
def predict_machine_failure(data):
    prediction = loaded_pipeline.predict(data)
    probability = loaded_pipeline.predict_proba(data)
    return prediction, probability

# Step 113: Test Prediction API Style
sample = X_test.iloc[:1]
pred, prob = predict_machine_failure(sample)

print(f"Sample Data Point Prediction: {pred[0]}")
print(f"Failure Probability: {prob[0][1]:.4f}")